<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_02_channel.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 · Notebook 02 — the 1-D channel

**Paired with L10.2 · Solid oxide cells**

Composition changes along the flow, so the local Nernst potential does too.
A 0-D model cannot represent this, and the error grows with utilisation.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## TODO 1 — a PINN for the channel

$$\frac{\partial c}{\partial t} + u\frac{\partial c}{\partial x}
= D\frac{\partial^2 c}{\partial x^2} + S(i)$$

The convection–diffusion operator of L9.1, with a reaction source set by the
local current density.

The channel coordinate is normalised to $[0, 1]$ — `pb.CHANNEL_DOMAIN` — the
same coordinate `pb.channel_profile` uses, so your solution and the reference
profile can be compared point for point.

In [ ]:
def residual_fn(model, xt, u_flow, D, source):
    c = model(xt)
    g = grad(c, xt)
    # TODO: return g[:, 1:2] + u_flow * g[:, 0:1] - D * d2(c, xt, 0) - source
    #
    # Column 0 is x and column 1 is t: time is the last column everywhere in
    # this course, so g[:, 1:2] is dc/dt and g[:, 0:1] is dc/dx.
    raise NotImplementedError

### Sampling and training

The samplers return **NumPy**, so that points can be plotted, saved and
checked without a device or a graph. Wrap them at the point of use:
`to_tensor(pts, requires_grad=True)` for anything you differentiate through,
`to_tensor(pts)` for everything else.

In [ ]:
# TODO: sample the space-time slab, build the loss, and train.
#
#   T_END = ...           # yours to choose; the channel is one unit long, so
#                         # 1 / u_flow is one residence time in these coordinates
#
#   xt = to_tensor(spacetime_points(2000, pb.CHANNEL_DOMAIN, t_span=(0.0, T_END),
#                                   seed=1), requires_grad=True)
#
#   The inlet is a single point in space, not an edge of a rectangle, so
#   boundary_points does not apply to a 1-D channel. Build it directly:
#
#       t_edge = np.linspace(0.0, T_END, 200).reshape(-1, 1)
#       xt_in  = to_tensor(np.concatenate([np.zeros_like(t_edge), t_edge], axis=1))
#       xt_ic  = to_tensor(initial_points(200, pb.CHANNEL_DOMAIN, t0=0.0, seed=2))
#
#   model = MLP(n_in=2, n_hidden=32, n_layers=4)
#   describe(model, 2000)
#
#   def loss():
#       f = residual_fn(model, xt, u_flow, D, source)
#       return mse(f) + mse(model(xt_in) - c_in) + mse(model(xt_ic) - c_0)
#
#   history = train_two_stage(model, loss, adam_steps=2000, lbfgs_steps=150)
#   plot_curves(history, title="1-D channel"); plt.show()
#
# Non-dimensionalise each residual before you choose a weight between the
# terms — the same argument as Ex_07.1 section 2.

raise NotImplementedError

## TODO 2 — quantify the 0-D error against utilisation

In [ ]:
for U in (0.2, 0.4, 0.6, 0.8):
    par = pb.SOCParams("SOEC", 1073.15, 0.10, 0.90, utilisation=U)
    x, pH2, pH2O, E = pb.channel_profile(par, 1.0)
    print(f"  utilisation {U:.1f}:  Nernst {E[0]:.3f} -> {E[-1]:.3f} V"
          f"   spread {E[-1]-E[0]:+.3f} V")
# TODO: at what utilisation does the spread exceed your voltage tolerance?

**Question.** A 0-D model uses the inlet composition. How large an error is that at 80% utilisation, and in which direction?